# Importing libraries and set style

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

import pickle

Style setting

In [ ]:
plt.style.use('seaborn-v0_8')
# plt.style.use('seaborn-v0_8-colorblind')
# sns.set_theme(style="whitegrid")

In [ ]:
# Figure
# plt.rcParams['figure.figsize'] = (8, 6)
# plt.rcParams['figure.dpi'] = 300

# Title
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

# Axes
plt.rcParams['axes.labelsize'] = 12

Font setting

In [ ]:
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Nimbus Sans"]

Import data

In [ ]:
df = pd.read_parquet("../raw_data/parquets/4-eumaco-spo2.parquet")

# with open('../raw_data/pickles/.pickle', 'rb') as file:
# 	df = pickle.load(file)

# Overview on all the data

In [ ]:
# df

In [ ]:
df.info()

# Plots

## Functions

In [ ]:
def line_plot(df, colname,
    title, x_label, y_label,
    output_dir, progr, output_name, output_format):


    original_len = len(df[colname])
    data = df[colname].dropna()
    na_count = original_len - len(data)

    per_year = data.dt.year.value_counts().sort_index()

    # Create figure
    fig, ax = plt.subplots()
    ax.plot(per_year.index, per_year.values)

    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)


    n_total = original_len
    n_valid = len(data)

    ax.text(
        0.98, 0.95,
        f'n = {n_valid} (of {n_total})\n{round((n_valid/n_total)*100)} %',
        transform=ax.transAxes,
        ha='right', va='top',
        fontsize=11
    )

    plt.tight_layout()
    plt.savefig(f"{output_dir}/{progr}-{output_name}.{output_format}", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def histbox_plot(df, colname,
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto'):
    
    original_len = len(df[colname])
    data = df[colname].dropna()
    na_count = original_len - len(data)

    if na_count > 0:
        print(f'Dropped {na_count} NA values ({na_count / original_len:.1%} of total)')

    # Create figure with 2 rows (hist + boxplot)
    fig, (ax1, ax2) = plt.subplots(
        2, 1, 
        figsize=figsize,
        sharex=True, 
        gridspec_kw={'height_ratios': [4, 1]}
    )

    # Histogram
    ax1.hist(data, bins=bins)
    ax1.set_ylabel(y_label)
    ax1.set_title(title)

    # Info about total and NAs
    n_total = original_len
    n_valid = len(data)

    ax1.text(
        0.98, 0.95,
        f'n = {n_valid} (of {n_total})\n{round((n_valid/n_total)*100)} %',
        transform=ax1.transAxes,
        ha='right', va='top',
        fontsize=11
    )

    # Boxplot
    ax2.boxplot(data, vert=False, widths=0.6)
    ax2.set_xlabel(x_label)
    ax2.set_yticks([])  # optional: hide y-axis ticks

    # Statistical info
    stats_info = {}
    # Outliers
    fliers = ax2.boxplot(data, vert=False, widths=0.6)['fliers'][0].get_xdata()
    # convert to Python int
    fliers = [int(v) for v in fliers]
    outlier_counts = dict(Counter(fliers))
    stats_info['outlier_counts'] = outlier_counts
    # Mean
    mean = round(data.mean(), 3)
    stats_info['mean'] = mean
    # stddev
    stddev = round(data.std(), 3)
    stats_info['stddev'] = stddev
    # Print
    print(f'Outliers: {outlier_counts}')
    print(f'Mean: {mean}')
    print(f'Std dev: {stddev}')

    # Savefig
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{progr}-{output_name}.{output_format}", dpi=300, bbox_inches='tight')
    plt.show()

### Common values

In [ ]:
output_dir = '../plots/4-eumaco-spo2'
# figsize = (8, 6)
progr = 0
output_format = 'png'

## Plot for wt field

In [ ]:
figsize = (8, 6)

### `tran.spo2.value`

In [ ]:
colname = 'tran.spo2.value'

In [ ]:
title = r'SpO$_2$ Distribution'
x_label = 'Percentage'
y_label = 'Count'
output_name = 'spo2'
progr += 1

In [ ]:
histbox_plot(df, colname,
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

### `tran.hear.value`

In [ ]:
colname = 'tran.hear.value'

In [ ]:
title = 'Heartrate values'
x_label = 'BPM'
y_label = 'Count'
output_name = 'heartrate'
progr += 1

In [ ]:
histbox_plot(df, colname,
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')